In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
from src.data_utils import get_returns_data

#gets the returns data frame
returns_df = get_returns_data()

#caluclates the mean returns for each asset
mean_returns = returns_df.mean()

#creates the covariance matrix between each of the assets
cov_matrix = returns_df.cov()

print(mean_returns)
print(cov_matrix)

The covariance matrix is a matrix that has the covariance between each pair of stocks. As you can notice this matrix is also symetric, similar to the correlation matrix. Additionally the diagonal elements is the variance of the stock.

The covariance matrix is important as portofolio risk doesn't just depend on the volatility of individual stocks but also how stocks move together. Using the covariance matrix we can calculate portfolio volatility. Now we will simulate a portfolio and calculate its return and volotility:

In [ ]:
import numpy as np

# creates 4 weights that sum to 1
weights = np.random.random(4)
weights = weights / np.sum(weights)

print(weights)
print(weights.sum())

In [ ]:
# calaculate expected portfolio return
portfolio_return = np.sum(mean_returns * weights)
print(portfolio_return)

# calculate portfolio volatility
portfolio_volatility = np.sqrt(
    np.dot(weights.T, np.dot(cov_matrix, weights))
)
print(portfolio_volatility)

To calculate the expected portfolio return you sum the product of the mean average return and the weighting of the asset. To calculate the volatility of the portfolio the formula is the square roots of the  product between the transpose of the weight vector and the product of the covariance matrix and the weight vector. 

The formula of portfolio volatility shows how diversification can help reduce risk. This is because if 2 stocks are negative correlated then the covariance between the 2 is negative meaning the portfolio volatility decreases.

In [ ]:
import matplotlib.pyplot as plt

# similating 10000 different portfolios with random weightings to each asset
num_portfolios = 10000

portfolio_returns = []
portfolio_volatilities = []
portfolio_weights = []

for i in range(num_portfolios):
    weights = np.random.random(4)
    weights = weights / np.sum(weights)

    portfolio_return = np.sum(mean_returns * weights)

    portfolio_volatility = np.sqrt(
        np.dot(weights.T, np.dot(cov_matrix, weights))
    )

    portfolio_returns.append(portfolio_return)
    portfolio_volatilities.append(portfolio_volatility)
    portfolio_weights.append(weights)


# create a data frame storing the portfolio return and portfolio volatility for each simulated portfolio
portfolio_results = pd.DataFrame({
    "Return": portfolio_returns,
    "Volatility": portfolio_volatilities,
    "Apple Weight": [w[0] for w in portfolio_weights],
    "Microsoft Weight": [w[1] for w in portfolio_weights],
    "Nvidia Weight": [w[2] for w in portfolio_weights],
    "S&P500 Weight": [w[3] for w in portfolio_weights]
})

portfolio_results.head()


# creates a scatter graph of the results
plt.figure(figsize=(10,6))

plt.scatter(portfolio_results["Volatility"], portfolio_results["Return"], s=10)

plt.title("Simulated Portfolio Risk vs Return")
plt.xlabel("Volatility")
plt.ylabel("Expected Daily Return")

plt.show()

The trend from this graph shows that as the expected daily return increases so does the volatility showing that there is increased risk associated with greater returns. 

In [ ]:
max_return_portfolio = portfolio_results.loc[portfolio_results["Return"].idxmax()]
min_volatility_portfolio = portfolio_results.loc[portfolio_results["Volatility"].idxmin()]

print("Highest return portfolio:")
print(max_return_portfolio)

print("Lowest volatility portfolio:")
print(min_volatility_portfolio)

From this result you can see that the portfolio with the highest return has most of its weight in Nvidia as it has the highest mean return out of the 4 assets. However this also increases the volatility cause Nvidia also has the highest volatility out of the 4 assets aswell. You can also see the portfolio with the lowest volatility has most of its weight in the S&P500 as the S&P500 has the lowest volatility out of the 4 assests but this comes with lower returns due to the S&P500 having the lowest mean return.

I'm now going to calulate the sharpe ratio to find the best balanced portfolio with respect the the mean return and volatility. In other words it calulates the return earned per unit of risk:

In [ ]:
#  calculates the sharpe ratio for every portfolio
portfolio_results["Sharpe Ratio"] = (
    portfolio_results["Return"]
    / portfolio_results["Volatility"]
)

#finds the portfolio with the greatest sharpe ratio
max_sharpe_portfolio = portfolio_results.loc[
    portfolio_results["Sharpe Ratio"].idxmax()
]

print(max_sharpe_portfolio)

The sharpe ratio is calculated by finding the difference between the portfolio return and risk free return divided by the portfolio volatility. For now I will assume that the risk free return is 0 so I can focus on comparing portfolios based on their risk-adjusted performance without introducing additional assumptions regarding interest rates. A higher sharpe ratio is better as it means you get more return per unit of risk. Therefor I have found the optimal portfolio adjusted with volatility.

In [ ]:
plt.figure(figsize=(10,6))

scatter = plt.scatter(
    portfolio_results["Volatility"],
    portfolio_results["Return"],
    c=portfolio_results["Sharpe Ratio"],
    cmap="viridis",
    s=10
)

plt.colorbar(label="Sharpe Ratio")

plt.xlabel("Volatility")
plt.ylabel("Expected Return")
plt.title("Portfolio Simulation")

plt.show()

This shows where the higher sharpe ratio portfolios lie on the graph. You can see that they are located on the top of the graph as if 2 portfolios have roughly the same volatility then the portfolio with a larger expected return will have a greater sharpe ratio.